<a href="https://colab.research.google.com/github/AnnabelleMcSharry/AI-ML-Car-Data/blob/main/associationrule.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install apyori
from apyori import apriori
import pandas as pd

In [ ]:
event = pd.read_csv('drug_adverse_event_dataset.csv')
print('Dimensions of dataset are :', event.shape)
print(event.head())

Dimensions of dataset are : (3500, 10)
   patient_id  age  gender      condition        drug_1        drug_2  \
0       20139   82  Female   Hypertension     Potassium  Atorvastatin   
1       21210   32  Female   Hypertension  Atorvastatin    Phenelzine   
2       21367   70    Male   Hypertension      Warfarin    Fluoxetine   
3       20242   26    Male  Heart Failure      Warfarin    Phenelzine   
4       20514   68  Female       Diabetes    Fluoxetine           NaN   

       drug_3        drug_4  num_medications  adverse_event  
0  Phenelzine           NaN                3              1  
1    Warfarin           NaN                3              0  
2  Phenelzine           NaN                3              1  
3   Metformin  Atorvastatin                4              1  
4         NaN           NaN                1              0  


In [ ]:
pd.set_option('display.max_rows', None)  #shows the count of all drug values
print(event["drug_1"].value_counts()) #high number in each category so i do not need to get rid of any drugs that will cause a very high confidence if there is a low number of occurences

drug_1
Metformin       422
Aspirin         401
Fluoxetine      395
Lisinopril      393
Potassium       392
Phenelzine      385
Warfarin        376
Atorvastatin    374
Amoxicillin     362
Name: count, dtype: int64


In [ ]:
#put the drugs into a single list for each record so they can be processed
drug_columns = ['drug_1', 'drug_2', 'drug_3', 'drug_4']
drug_list = []

for index, row in event.iterrows():
    drugs_in_row = [str(row[col]) for col in drug_columns if pd.notnull(row[col])]
    if drugs_in_row:
        drug_list.append(drugs_in_row)
    else:
        drug_list.append([]) # Append an empty list if no drugs are found for the row

event['drug_list'] = drug_list

# Drop the original drug columns
event = event.drop(columns=drug_columns)

print(f"Number of drug_list: {len(drug_list)}")
print("First 5 drug_list:")
for t in drug_list[:5]:
    print(t)

print("\nUpdated DataFrame head:")
print(event.head())

Number of drug_list: 3500
First 5 drug_list:
['Potassium', 'Atorvastatin', 'Phenelzine']
['Atorvastatin', 'Phenelzine', 'Warfarin']
['Warfarin', 'Fluoxetine', 'Phenelzine']
['Warfarin', 'Phenelzine', 'Metformin', 'Atorvastatin']
['Fluoxetine']

Updated DataFrame head:
   patient_id  age  gender      condition  num_medications  adverse_event  \
0       20139   82  Female   Hypertension                3              1   
1       21210   32  Female   Hypertension                3              0   
2       21367   70    Male   Hypertension                3              1   
3       20242   26    Male  Heart Failure                4              1   
4       20514   68  Female       Diabetes                1              0   

                                         drug_list  
0            [Potassium, Atorvastatin, Phenelzine]  
1             [Atorvastatin, Phenelzine, Warfarin]  
2               [Warfarin, Fluoxetine, Phenelzine]  
3  [Warfarin, Phenelzine, Metformin, Atorvastatin]  
4  

In [ ]:
columns_to_check_duplicates = [col for col in event.columns if col != 'drug_list']
duplicates = event[columns_to_check_duplicates].duplicated()
print(f"Number of duplicate rows (excluding 'drug_list' column): {duplicates.sum()}")
#the duplicated function cannot process a python list so I did not include that but this shows there are no duplicated patients so all of the rows should eb kept

# Check for missing data
#do not want to delete 669 rows of data but can still use the rows if i do not use the conditions column in my rule association
missing_data = event.isnull().sum()
print("\nMissing data in each column:")
print(missing_data)

Number of duplicate rows (excluding 'drug_list' column): 0

Missing data in each column:
patient_id           0
age                  0
gender               0
condition          669
num_medications      0
adverse_event        0
drug_list            0
dtype: int64
